# Liberias

In [61]:
%pip install numpy
%pip install matplotlib
%pip install pandas
%pip install scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys 

from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [191]:
# Global Values
ALPHAS = np.logspace(-3,3,7) # for the alpha for RidgeCV
np.random.seed(8) # for reproducibility

## This is for the train data
N_narx1 = 1000
N_narx2 = 1000
split1 = int(N_narx1*0.5)
split2 = int(N_narx2*0.5)

# Metricas

In [192]:
def nmse(y_t, y_p): # professor formula check both outputs
    yt_bar = y_t.mean(axis=0)
    num = np.sum((y_t - y_p)**2)
    den = np.sum((y_t - yt_bar)**2)
    return num/den

def rmse(y_t, y_p):
    assert  y_t.shape[1] == 2
    N_ts= y_t.shape[0]

    total_e= np.sum((y_t - y_p)**2)
    return np.sqrt(total_e / (2 * N_ts))

# Graficas

In [193]:
def plot_1(array, title="Signal Plot", label="Data"):
    plt.figure(figsize=(10, 4))
    plt.plot(array, label=label, color='blue')
    plt.title(title)
    plt.xlabel("Time (k)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [194]:
def plot_comparison(Y_true, Y_pred, title=""):
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    for i, ax in enumerate(axes):
        ax.plot(Y_true[:, i], label="Real", linewidth=1)
        ax.plot(Y_pred[:, i], label="Predicted", linewidth=1, linestyle="--")
        ax.set_ylabel(f"y{i+1}")
        ax.legend(fontsize=8)
    axes[0].set_title(title)
    plt.tight_layout()
    plt.show()


# NARX

## NARX1

$$y_1(k+1) = 0.5\,y_2(k-1) + \sin(y_2(k)) + 0.3\,u(k-1) + w_1(k+1)$$
$$y_2(k+1) = 0.5\,y_1(k-1) + \sin(y_1(k)) + 0.2\,u(k) + w_2(k+1)$$

In [195]:
## one input = u1 and u2, two oputs = y1 and y2
def NARX1 (u,sigma= 0.01):
    ## Starting variables
    N = len(u)
    w1 = np.random.normal(0, sigma, N)
    w2 = np.random.normal(0, sigma, N)
    y1 , y2 = np.zeros(N), np.zeros(N)
    
    # Calculating the outputs
    for k in range(2, N):
      
        y1[k] = (0.5 * y2[k-2].item()) + (np.sin(y2[k-1].item())) + (0.3 * u[k-2].item()) + w1[k].item()
        y2[k] = (0.5 * y1[k-2].item()) + (np.sin(y1[k-1].item())) + (0.2 * u[k-1].item()) + w2[k].item()
    
    return np.column_stack([y1,y2])
# Return of this function 
# is a 2D array of shape (N, 2) where the first column corresponds to y1 and the second column corresponds to y2.

In [196]:
rng= np.random.default_rng(0)
narx1_data = rng.uniform(-1,1,N_narx1)

narx1_u = narx1_data.reshape(-1, 1) # Reshape 
narx1_y = NARX1(narx1_u, sigma=0.01) # Generate the output data using the NARX1 function

narx1_u_tr , narx1_u_ts = narx1_u[:split1], narx1_u[split1:]
narx1_y_tr, narx1_y_ts = narx1_y[:split1], narx1_y[split1:]

plot_1(narx1_u, title="Input Signal (u)", label="u")
plot_1(narx1_y[:,0], title="Output Signal(y1)", label="y1")
plot_1(narx1_y[:,1], title="Output Signal (y2)", label="y2")

### NARX2
Is a system with two outputs and a two input.
$$y_1(k+1) = \frac{y_1(k)\,y_1(k-1)\,y_1(k-2)\,(y_1(k-2)-1)\,u_2(k-1) + u_2(k)}{1 + y_2(k-1)^2 + y_2(k-2)^2} + w_1$$

$$y_2(k+1) = \frac{y_2(k)\,y_2(k-1)\,y_2(k-2)\,(y_2(k-2)-1)\,u_1(k-1) + u_1(k)}{1 + y_1(k-1)^2 + y_1(k-2)^2} + w_2$$

In [197]:
## two inputs = u1 and u2, two oputs = y1 and y2
def NARX2 (u1, u2, sigma = 0.01): 
    
    ## Starting variables 
    N = len(u1)
    w1 = np.random.normal(0, sigma, N)
    w2 = np.random.normal(0, sigma, N)
    y1,y2 = np.zeros(N) , np.zeros(N)

    ## NARX2 has a tendency to explode... there is something that can control this explosition 
    # np.clip function that can be used to limit the values of y1 and y2 to a certain range. 
    CLIP = 5.0 # This is a conservative clip to keep the simulation bounded. It can be adjusted based on the expected range of the outputs.

    for k in range(3, N):
      
        y1_1, y1_2, y1_3 = y1[k-1].item(), y1[k-2].item(), y1[k-3].item()
        y2_1, y2_2, y2_3 = y2[k-1], y2[k-2], y2[k-3]
        
        num1 = (y1_1 * y1_2 * y1_3) * (y1_3 - 1) * u2[k-2].item() + u2[k-1].item()
        num2 = (y2_1 * y2_2 * y2_3) * (y2_3 - 1) * u1[k-2].item() + u1[k-1].item()
        
        den1 = 1 + (y2_2**2) + (y2_3**2)
        den2 = 1 + (y1_2**2) + (y1_3**2)
        
        y1[k] = np.clip((num1 / den1) + w1[k],-CLIP,CLIP)
        y2[k] = np.clip((num2 / den2) + w2[k],-CLIP,CLIP)
       

    return y1,y2

In [198]:
rng = np.random.default_rng(0)
narx2_u1_data = rng.uniform(-0.5, 0.5, N_narx2)
narx2_u2_data = rng.uniform(-0.5, 0.5, N_narx2)

narx2_U = np.column_stack([narx2_u1_data, narx2_u2_data])   # shape (N, 2)
y1 ,y2 = NARX2(narx2_U[:,0], narx2_U[:,1], sigma=0.01)
narx2_Y = np.column_stack([y1, y2])                          # shape (N, 2)

narx2_U_tr, narx2_U_ts = narx2_U[:split2], narx2_U[split2:]
narx2_y_tr, narx2_y_ts = narx2_Y[:split2], narx2_Y[split2:]

plot_1(narx2_U[:, 0], title="Input Signal (u1)", label="u1")
plot_1(narx2_U[:, 1], title="Input Signal (u2)", label="u2")
plot_1(narx2_Y[:, 0], title="Output Signal (y1)", label="y1")
plot_1(narx2_Y[:, 1], title="Output Signal (y2)", label="y2")

## Task 2. Machine Learning Stategy
Since our program works with regression NARX. We make a fuction to build the features used in the training.

In [199]:
def build_mismo_features(U, Y, d, na, s):
    """
    Feature en fila k:
      [Y(k-d), ..., Y(k-d-na)]   <- (na+1)*2 outputs pasados
      [U(k+1), ..., U(k+s)]      <- s * n_inputs futuros CONOCIDOS
    Target:
      [Y(k+1), ..., Y(k+s)]      <- s*2 aplanados
    """
    U, Y = np.asarray(U, float), np.asarray(Y, float)
    N, n_in= U.shape
    X_rows, T_rows = [], []
    
    for k in range(d + na, N - s):
        #print(f"\n--Building features for row {k}--")
        feat = [] 
        for lag in range(d, d + na + 1):
            feat.extend(Y[k - lag])
            #print(f"    - Adding Y(k-{lag}) to features: {Y[k - lag]}")
        for h in range(1, s + 1):
            feat.extend(U[k + h])
            #print(f"    - Adding U(k+{h}) to features: {U[k + h]}")
        X_rows.append(feat)
        #print(f"  Built features for row {k}: {feat}")
        tgt = []
        for h in range(1, s + 1):
            tgt.extend(Y[k + h])
            #print(f"    - Adding Y(k+{h}) to target: {Y[k + h]}")
        T_rows.append(tgt)

    return np.array(X_rows), np.array(T_rows)

In [200]:
def train_mismo(U_tr, Y_tr, d, na, s):
    #print(f"\n - Training MISMO with d={d}, na={na}, s={s}...")
    
    #print("   Building features...")
    X, T = build_mismo_features(U_tr, Y_tr, d, na, s)

    #print("   Training model...")
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))
    model.fit(X, T)
    return model

In [201]:
def mismo_rollout(model, U_test, d, na, s, y_prefix=None):
    """
    Predicción de horizonte largo con MISMO.
    Avanza en bloques de s pasos; el último puede ser menor.
    """
    Nts = len(U_test)
    n_in = U_test.shape[1]
    Y_hat = np.zeros((Nts, 2))

    buf_len = d + na + 1
    buf = np.zeros((buf_len, 2))
    
    if y_prefix is not None:
        take = min(buf_len, len(y_prefix))
        buf[-take:] = y_prefix[-take:]

    k = 0
    while k < Nts:
        block = min(s, Nts - k)

        feat = []
        for lag in range(d, d + na + 1):
            feat.extend(buf[buf_len - 1 - lag])
        for h in range(1, s + 1):
            ui = k + h - 1
            feat.extend(U_test[ui] if ui < Nts else np.zeros(n_in))

        pred_flat = model.predict(np.array(feat, float).reshape(1, -1))[0]
        pred_block = pred_flat.reshape(s, 2)

        Y_hat[k:k + block] = pred_block[:block]
        for step in range(block):
            buf = np.roll(buf, -1, axis=0)
            buf[-1] = Y_hat[k + step]

        k += block

    return Y_hat

# Escoger los mejores parametros 
tenemos una funcion para calcualr los parametros como d na nb/s , s para el caso de MISMO

## 1. Escoger el d
We use cross-correlation to get an initial range for delay $d$.

- Keep only causal lags ($k \geq 0$).
- Use pairwise CCF peaks as a guide, then validate with grid search.

In [202]:
def cross_correlation(y, u, max_lag=40):
    y_c = y - y.mean()
    u_c = u - u.mean()
    full_corr = np.correlate(y_c, u_c, mode='full')
    N = len(y)
    lags = np.arange(-(N - 1), N)

    norm = np.sqrt(np.sum(y_c**2) * np.sum(u_c**2))
    full_corr = full_corr / (norm + 1e-12)

    mask = (lags >= -max_lag) & (lags <= max_lag)
    return lags[mask], full_corr[mask]

In [203]:
def estimate_d_candidates(Utr,Ytr,max_lag=40,max_d=8):
    nu , ny = Utr.shape[1], Ytr.shape[1]
    peak_lags = {}
    
    for i in range(nu):
        for j in range(ny):
            lags, corr = cross_correlation(Ytr[:, j], Utr[:, i], max_lag=max_lag)
            pos_mask = lags >= 0
            peak_pos = int(lags[pos_mask][np.argmax(np.abs(corr[pos_mask]))])
            peak_lags[(i, j)] = peak_pos

    print('\nPositive-lag peaks per (input, output) pair:')
    for (i, j), lag in peak_lags.items():
        print(f'  u{i+1} -> y{j+1}: peak at lag {lag}')

    peak_values = np.array(list(peak_lags.values()),dtype=int)
    d_min = int(np.min(peak_values))
    d_med = int(np.median(peak_values))

    baseline = set(range(0, 5))
    around_med = set(range(max(0, d_med - 2), d_med + 4))
    around_min = set(range(max(0, d_min - 1), d_min + 3))
    d_candidates = sorted(d for d in (baseline | around_med | around_min) if 0 <= d <= max_d)

    d_candidates = sorted(
        d for d in (baseline | around_med | around_min) if 0 <= d <= max_d
    )
    print(f'\nCCF min peak: {d_min}, median peak: {d_med}')
    print(f'\nWill test d in {d_candidates} during grid search')

    return d_candidates, peak_lags

In [204]:
d1, peaks1 = estimate_d_candidates(narx1_u_tr, narx1_y_tr, max_lag=40,max_d=8)
d2, peaks2 = estimate_d_candidates(narx2_U_tr, narx2_y_tr, max_lag=40,max_d=8)

grid1 = {
    'd': d1,
    'na': [0,1,2,3,4,5,6,7,8,9,10],
    's': [1,5,10,20,50,100,200]
}

grid2 = {
    'd': d2,
    'na': [0,1,2,3,4,5,6,7,8,9,10],
    's': [1,5,10,20,50,100,200]
}

In [205]:
def grid_search(U_tr, Y_tr, U_ts, Y_ts, grid, y_prefix=None, label=""):
    
    rows = []

    for params in ParameterGrid(grid):
        d = int(params['d'])
        na = int(params['na'])
        s = int(params['s'])

        mod = train_mismo(U_tr, Y_tr, d, na, s)
        yhat = mismo_rollout(mod, U_ts, d, na, s)

        rows.append({
            'd': d,
            'na': na,
            's': s,
            'nmse': nmse(Y_ts, yhat),
            'rmse': rmse(Y_ts, yhat)
        })
    
    if not rows:
        return {}, pd.DataFrame()
    
    df = pd.DataFrame(rows).sort_values(['nmse','rmse']).reset_index(drop=True)
    
    print(f"\n=== Grid (d, na, s) — {label} ===")
    print(df.to_string(index=False))

    best_row = df.iloc[0]
    best_params = {
        'd':  int(best_row['d']),
        'na': int(best_row['na']),
        's':  int(best_row['s']),
    }
    print(f"\nMejor combinación: d={best_params['d']}, "
          f"na={best_params['na']}, s={best_params['s']}  "
          f"(nmse={best_row['nmse']})")

    return best_params, df

In [206]:
## This code take some time to run, be patient.
print("Best Parameters for NARX1:")
best_n1, df_n1 = grid_search(narx1_u_tr, narx1_y_tr, narx1_u_ts, narx1_y_ts,grid1, y_prefix=narx1_y_tr, label="NARX1")

# split 0.7
# Best Parameters for NARX1:
# Mejor combinación: d=4, na=9, s=100  (nmse=0.07142388127847606)

In [ ]:
## This code take some time to run, be patient.
print("Best Parameters for NARX2:")
best_n2, df_n2 = grid_search(narx2_U_tr, narx2_y_tr, narx2_U_ts, narx2_y_ts,grid2, y_prefix=narx2_y_tr, label="NARX2")

# split 0.7
# Best Parameters for NARX2:
# Mejor combinación: d=8, na=9, s=100  (nmse=0.01994511474098388)

---

In [ ]:
narx1_modelMISMO = train_mismo(narx1_u_tr, narx1_y_tr, best_n1['d'], best_n1['na'], best_n1['s'])
narx1_yhat = mismo_rollout(narx1_modelMISMO , narx1_u_ts, best_n1['d'], best_n1['na'], best_n1['s'], y_prefix=narx1_y_tr)

print(f"NARX 1 | MISMO (s={best_n1['s']}) NMSE={nmse(narx1_y_ts,narx1_yhat):.5f}" f"| Recursive NMSE={nmse(narx1_y_ts,narx1_yhat):.5f}")
plot_comparison(narx1_y_ts, narx1_yhat, title=f"NARX1 - MISMO Prediction (s={best_n1['s']})")

In [ ]:
narx2_modelMISMO = train_mismo(narx2_U_tr, narx2_y_tr, best_n2['d'], best_n2['na'], best_n2['s'])
narx2_yhat = mismo_rollout(narx2_modelMISMO , narx2_U_ts, best_n2['d'], best_n2['na'], best_n2['s'], y_prefix=narx2_y_tr)

print(f"NARX 2 | MISMO (s={best_n2['s']}) NMSE={nmse(narx2_y_ts,narx2_yhat):.5f}" f"| Recursive NMSE={nmse(narx2_y_ts,narx2_yhat):.5f}")
plot_comparison(narx2_y_ts, narx2_yhat, title=f"NARX1 - MISMO Prediction (s={best_n2['s']})")

# Task 2.c. Baseline Approach (1p)

#### Step 1: Defining the Outputs and Features vectors
To prepare the data from the aboce datframe for training, for a given row $k$, we first need to extract the target vector $y(k)$ and the features vector $X(k)$ based on the optimal parameters $n_a, n_b$ and $d$ found before. 

**1. The target vector $y(k)$**

The target vector represents our ground-truth values for the next time step: 

$$
y[k] = \left( y_1(k+1), \ y_2(k+1) \right)
$$

This is is easily obatined from the original dataframe by shifting the original outputs columns by one : 

In [ ]:
y = df[['y1(k)', 'y2(k)']].shift(-1).rename(columns={'y1(k)': 'y1(k+1)','y2(k)': 'y2(k+1)'})
y = y.iloc[:-1].reset_index(drop=True)
print(y)

#### Step 2: The feature vector/matrix ($X$)
The features vector $X(k)$ is constructed from the lagged observations of both inputs and ouputs. In `pandas`, these are generated by properly shifting the original columns down (e.g., $y_1(k-4)$ is the values of  $y_1(k-4)$ from $4$ rows below). Our goal is to obtain a feqture vector defined qs : 

$$
X[k] = \left( y_1(k-d), \dots, y_1(k-d-n_a), \ y_2(k-d), \dots, y_2(k-d-n_a), \ u_1(k), \dots, u_1(k-n_b), \ u_2(k), \dots, u_2(k-n_b) \right).
$$

However, beforte obtqining the feqture mqtrix and vectors by shifting the original dataframe accordingly, given that we dsicarded the last row of the ouputs daframe (the row of index $1000$), the same has to be done with the last row of the data. Indeed, for examples, the values of $u_1(1000)$ and $u_2(1000)$ will never be used to compute the next predicitons. 

In [ ]:
data = df.iloc[:-1].reset_index(drop=True)
data

Now, to obtain the features dataframe from the above dataframe, we ned to properly shift everyhting following the best $n_a, n_b$ and $d$ found before in order to have every releavnt features fro predidciting $y_1(k+1)$ and $y_2(k+1)$ on row $k$. To do this, let us create an function `create_lagged_features` that takes the above dataframe and the parmatersd as arguments, and returns a dataframe with all the releavant features to compute $y_1(k+1)$ and $y_2(k+1)$ on row $k$.

`Note` : Shifting the dataframe only following this method would produce starting NaNs values in the first few rows of the returned daframe. Indeed for example if $d = 5$, on row $k = 3$ we would end up with $y_1(k-d) = y_1(3-5) = y_1(-2)$, which is obviously a NaN. To paliate this issue and following project requirements, we are going to fill all these starting NaNs values in the dataframe with zeros.

In [ ]:
def create_lagged_features(df: pd.DataFrame, n_a: int, n_b: int, d: int) -> pd.DataFrame:

    # Create a copy of the original dataframme
    dataframe = df.copy()

    # Initialize the lags for y1 and y2 from scratch
    y1_lags = []
    y2_lags = []

    # For the outputs y1 and y2, we need to loop from y(k-d) to y(k-d-n_a)
    for i in range(d, d + n_a + 1):

        y1_col, y2_col = f'y1(k-{i})', f'y2(k-{i})'

        # Shift the datframe accoridngly from the origi,ql columns
        dataframe[y1_col] = dataframe['y1(k)'].shift(i)
        dataframe[y2_col] = dataframe['y2(k)'].shift(i)

        # Append the results to the lags 
        y1_lags.append(y1_col)
        y2_lags.append(y2_col)

    # For the inputs, we want to keep the current values 
    u1_lags = ['u1(k)']
    u2_lags = ['u2(k)']

    # Here, we need to loop from u(k-1) to u(k-n_b)
    for i in range(1, n_b + 1):

        u1_col, u2_col = f'u1(k-{i})', f'u2(k-{i})'

        dataframe[u1_col] = dataframe['u1(k)'].shift(i)
        dataframe[u2_col] = dataframe['u2(k)'].shift(i)

        # Append the results to the lags
        u1_lags.append(u1_col)
        u2_lags.append(u2_col)

    # Create a proper order for the columns in the retruned dataframe
    columns_order = y1_lags + y2_lags + u1_lags + u2_lags

    # As explained in the note from the above cell, we fill all the starting Nans values with zeros
    dataframe[columns_order] = dataframe[columns_order].fillna(0.0)

    result = dataframe[columns_order]

    return result

In [ ]:
# The feqtures datafrme is simply obtained by calling the function with the datafrme and the parmaters values.
X = create_lagged_features(data, na, nb, d)
X.head(10)

#### Step 3: Training non-deep learning models
Our objective when training models is to identify the optimal mapping functions that minimize speicif evqluations metrics, such as NMSE or RMSE. 

Concretely, we define the feature matrix/vector $X(k)$ and we aim to estimate functions $\hat{f}_1$ and $\hat{f}_1$ such that :

$$
\hat{y}_1(k+1) = \hat{f}_1\left( X(k) \right), \quad \hat{y}_2(k+1) = \hat{f}_2\left( X(k) \right)
$$

The goal is to ensure that these predicitons are as close as possible to the ground-truth values $y_1(k+1)$ and $y_2(k+1)$ from the training data, based on our cvhosen metrics. 

Then, we note that when we are going to train our sklearn models using the `fit` functiuon and then obtain predictions using `predict`, this will be done in a One-Step-Ahead way : The p^redicitons are based on the previous ground-truth values. 

However, givne the context of the project, this is not what we want. Indeed, what we need is the model to make the next predicition by feeding its own past predicitons back into the model. In the context of the test set, we won't any ground-truth reference value like we would with the training data.

Lret us tehrefore define a `multiple_steps_ahead` function to feed back to the model its own previous preicitons.


In [ ]:
def multiple_steps_ahead(model, X_val, n_a: int, d:int) -> np.ndarray:

    # Initialize the total number of steps corrsponding to the feature matrix size
    n_steps = len(X_val)

    # Initialize all the predicitons to zeros
    predictions = np.zeros((n_steps, 2))

    # Make a copy of the validation data
    X_sim = X_val.copy()

    # Foudn the indices of the validation set in teh case of validation 
    # Else the start of the dataframe in test mode 
    val_indices = X_val.index

    # Loop over all the predicitons to feed them back to the model
    for k in range(n_steps):
        
        # Get the index in the datafrme
        actual_idx = val_indices[k]

        for lag_val in range(d, d + n_a + 1):

            pred_idx = k - lag_val - 1

            if pred_idx >= 0:

                y1_col = f'y1(k-{lag_val})'
                y2_col = f'y2(k-{lag_val})'

                # If the columns already exist, add the previous prdicitons
                if y1_col in X_sim.columns:
                    X_sim.at[actual_idx, y1_col] = predictions[pred_idx, 0]

                if y2_col in X_sim.columns:  
                    X_sim.at[actual_idx, y2_col] = predictions[pred_idx, 1]

        # Once all the previous predictions have been added/the dataframe was modified
        # We can use tem to predict the next one using the updated row
        current_prediciton = model.predict(X_sim.loc[[actual_idx]])

        # Add the current prediciton to the list
        predictions[k] = current_prediciton[0]

    # Once the loo^p is obver we can return the all the precitions
    return predictions

#### Step 4: Model Training and Cross-Validation

Having found our target and features vectors and with our tuility fucntions defined, we now proceed to the training and evaluation phases.

##### Time-Series Cross Validation 

Our systrem is inherently time dependent. Because of that, standard k-fold cross validation is inappropirate as it would involve prediciting past values using future data. Instead, we are going to use the `TimeSeriesSplit` method from `sklearn.model_selection`. This validation approach ensure that the training set always precedes the validation set, maintaining the casual integrity of the times series and preventing data leakage. 

##### Candidate Models

In what follows, we are going to evaluate three disticnt non deep-learning models to identify the best fit for our system : 

1. **Linear Regression (OLS)** : It will serves as our baseline, assuming a linear relationship between the lagged features and the outputs. 
2. **K-Nearest Neighbors (KNN)** : Predicts outputs based on teh similarity between the current feature vector and past training examples. 
3. **Random Forest** : Use of mutliple decision trees to capture non-linear interactions between features and outputs. 

##### Evaluation Strategy

For each fold of the time series cross-validation, each model will be evaluated by computing the RMSE and NMSE metrics on the validation part of the fold using two methods : 

- One-Step-Ahead (OSA) : As expalined above, predicitons are made using groud-truth lagged values. It will measure the model's immediate predictive accuracy. 
- Multiple-Steps-Ahead (MSA) : Will measure how well the model reacts to its own previous pre-dicitons being used as ground-truth. 

Given qll these considerations, it is now time to finally train our models.

In [ ]:
# Initialize the time series split 
tscv = TimeSeriesSplit(n_splits=5)

# Initialize the three models with random parameters
models = {
    "OLS": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "RF": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Initialize dictionnaries top store the results for NMSE and RMSE in bothg OSA and MSA
osa_results = {name: [] for name in models}
msa_results = {name: [] for name in models}

# The cross-validation loop on the time series
for i, (train_index, val_index) in enumerate(tscv.split(X)):

    # For each fold, we have a training set and a validation set. 
    # The TimeSeriesSplit allow to easily get these indices and obtain the corrsponding features and outputs values.
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # For clairty, we can implement a small print to see exactly what happens inside each fold
    #print(f"--- Fold {i+1} --- Training set size : {len(X_train)} --- Validation set size : {len(X_val)} ---")

    # For each fold, we can now train our models and evaluate the RMSE and NMSE for OSA and MSA
    for name, model in models.items():

        # Training the model on the training set of the fold
        model.fit(X_train, y_train)

        # One-Step-Ahead predicitons on the validation set
        osa_pred = model.predict(X_val)
        rmse_osa = rmse(y_val.values, osa_pred)
        nmse_osa = nmse(y_val.values, osa_pred)
    
        osa_results[name].append((rmse_osa, nmse_osa))

        # Multiple-Steps-Ahead predicitons on the validation set
        # Note : This is done by using the multiple_steps_ahead functions previoulsy defined
        msa_pred = multiple_steps_ahead(model, X_val, na, d)
        rmse_msa = rmse(y_val.values, msa_pred)
        nmse_msa = nmse(y_val.values, msa_pred)

        msa_results[name].append((rmse_msa, nmse_msa))


In [ ]:
 # Print the results fro the current fold and model
print("--- Final Average performance ---")
for name in models: 

    osa_rmses = [s[0] for s in osa_results[name]]
    msa_rmses = [s[0] for s in msa_results[name]]
    osa_nmses = [s[1] for s in osa_results[name]]
    msa_nmses = [s[1] for s in msa_results[name]]

    # Compute and print the mean and the std for RMSE and NMSE 
    # For each model, and for both OSA and MSA
    print(f"{name:12} OSA: RMSE = {np.mean(osa_rmses):.4f} (+/- {round(np.std(osa_rmses), 3)}), NMSE = {np.mean(osa_nmses):.4f} (+/- {round(np.std(osa_nmses), 3)}) ")
    print(f"{name:12} MSA: RMSE = {np.mean(msa_rmses):.4f} (+/- {round(np.std(msa_rmses), 3)}), NMSE = {np.mean(msa_nmses):.4f} (+/- {round(np.std(msa_nmses), 3)}) ")

#### Step 5: Optimizacion Strategy
The previous analysis provided a first overiew of the situiation but, to ensure we are evaluating the full potential of our candidate modelsd, we now perfom a grid search with cross-validation using the class `GridSearchCV` from `sklearn.model_selection`. This process will iterates through a predefined set of hyperparameters, selecting the combiantion taht minimizes both the RMSE and NMSE accross the time series folds.

- For the Random Forest, we are going to tune the number of trees in the forest (`n_estimator`), th emaximum depth of the trees (`max_depth`), and the minimum number of samples required at a leaf node (`min_samples_lead`). This will help balance underfitting and overfitting (variance vs bias).

- For KNN, we optimize the number of neighbors (`n_neighbors`). A smaller $k$ makes the model sensitive to noise, while a larger one can make it too smooth to capture fast dynamic system.

In [ ]:
# Compute an average metric to use as the scorer 
def average_metrics(y_true, y_pred):
    return ((rmse(y_true.values, y_pred) + nmse(y_true.values, y_pred)) / 2 )

# This might need to be checked 

# WE want to use a custom scorer based on the average_metrics function
custom_scorer = make_scorer(average_metrics, greater_is_better=False)

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200, 300, 500],
        'max_depth': [None, 5, 10, 20],
        'min_samples_leaf': [1, 2, 5],
    },
    cv=tscv,
    scoring=custom_scorer,
    n_jobs=1,
    verbose=1,
)

rf_grid.fit(X, y)

print(f"Best RF params : {rf_grid.best_params_}")
print(f"Best RF CV Averaged metric : {-rf_grid.best_score_:.4f}")

knn_grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid={'n_neighbors': [3,5,7,10,15,20]},
    cv=tscv,
    scoring=custom_scorer,
    n_jobs=1,
)

knn_grid.fit(X, y)

print(f"Best KNN params : {knn_grid.best_params_}")
print(f"Best KNN CV Averaged metric : {-knn_grid.best_score_:.4f}")

best_rf = rf_grid.best_estimator_
best_knn = knn_grid.best_estimator_

### Task 2.d. Deep Learning Approach (2p)

## Task 3: Deployment on Real Dataset (1.5p)

### 3.1. Load Real Dataset

### 3.2. Parameter Estimation on real Data

### 3.3. Final Model Training 

### 3.4. Generate Predictions $\bar{Y_1}$ and $\bar{Y_2}$

## Task 4. Submission

## Ranking Score


In [ ]:
# 1. Load Data
seed = 8 # lucky number 
np.random.seed(seed)

## dataset path
if (Path.cwd() / 'src').exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

np.set_printoptions(suppress=True, precision=4)

dataset_path = PROJECT_ROOT / 'data/raw/StudentdataNARX.npz'
if not dataset_path.exists():
    raise FileNotFoundError(f'Dataset not found: {dataset_path}')

data = np.load(dataset_path)
required_keys = ['Utr', 'Ytr', 'Uts1', 'Uts2']
missing = [k for k in required_keys if k not in data.files]
if missing:
    raise KeyError(f'Missing arrays in dataset: {missing}')

# TRAIN DATA
Utr = np.asarray(data['Utr'], dtype=float)
Ytr = np.asarray(data['Ytr'], dtype=float)

t_u1 = Utr[:,0]
t_u2 = Utr[:,1]
t_y1 = Ytr[:,0]
t_y2 = Ytr[:,1]

# train in data frame
df = pd.DataFrame({
    'u1(k)': Utr[:,0],
    'u2(k)': Utr[:,1],
    'y1(k)': Ytr[:,0],
    'y2(k)': Ytr[:,1]
})

print(df.describe())

## TEST DATA
Uts1= np.asarray(data['Uts1'], dtype=float)
Uts2= np.asarray(data['Uts2'], dtype=float)

te_u1_i1 = Uts1[:,0]
te_u1_i2 = Uts1[:,1]
te_u2_i1 = Uts2[:,0]
te_u2_i2 = Uts2[:,1]

# 2. Add Noise 
var= 0.01 #  We choose this value
N = t_u1.size
sigma = np.sqrt(var)
w1 = np.random.normal(loc=0,scale=sigma,size=N)
w2 = np.random.normal(loc=0,scale=sigma,size=N)